In [17]:
import pandas as pd
import mysql.connector

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [18]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Kiphire@2019",
    database="HR_PROJECT"
)

df = pd.read_sql("SELECT * FROM employees_final", conn)

df.head()

/var/folders/5k/_k6qfv4x075_lckbwr4hpvw40000gn/T/ipykernel_1421/2208579207.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM employees_final", conn)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,risk_level,attrition_flag,distance_category
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,8,0,1,6,4,0,5,Low Risk,1,Near
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,10,3,3,10,7,1,7,Low Risk,0,Medium
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,7,3,3,0,0,0,0,Low Risk,1,Near
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,8,3,3,8,7,3,0,Low Risk,0,Near
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,6,3,3,2,2,2,2,Low Risk,0,Near


In [19]:
columns_to_keep = [
    'EmployeeNumber','Age','Gender','Department','JobRole','MaritalStatus',
    'MonthlyIncome','DistanceFromHome','YearsAtCompany','WorkLifeBalance',
    'JobSatisfaction','Attrition','attrition_flag',
    'risk_level','distance_category'
]


In [20]:
df = df[columns_to_keep]

In [25]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ==============================
# 2. LOAD DATA
# ==============================


# ==============================
# 3. SELECT CLEAN FEATURES
# ==============================
columns_to_keep = [
    'EmployeeNumber','Age','Gender','Department','JobRole','MaritalStatus',
    'MonthlyIncome','DistanceFromHome','YearsAtCompany','WorkLifeBalance',
    'JobSatisfaction','attrition_flag',
    'risk_level','distance_category'
]

df = df[columns_to_keep]

# ==============================
# 4. ENCODE CATEGORICAL DATA
# ==============================
categorical_cols = [
    'Gender','Department','JobRole','MaritalStatus',
    'risk_level','distance_category'
]

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# ==============================
# 5. DEFINE X & y
# ==============================
X = df.drop(['attrition_flag'], axis=1)
y = df['attrition_flag']

# ==============================
# 6. TRAIN-TEST SPLIT
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ==============================
# 7. TRAIN MODEL (BALANCED)
# ==============================
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

# ==============================
# 8. THRESHOLD TESTING
# ==============================
probs = model.predict_proba(X_test)[:,1]

print("🔍 Threshold Tuning Results:\n")

for t in [0.4, 0.5, 0.6]:
    y_pred = (probs > t).astype(int)
    print(f"\n===== Threshold: {t} =====")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

# ==============================
# 9. FINAL THRESHOLD (SET HERE)
# ==============================
FINAL_THRESHOLD = 0.5   # change based on results above

# ==============================
# 10. FINAL EVALUATION
# ==============================
y_final = (probs > FINAL_THRESHOLD).astype(int)

print("\n🎯 FINAL MODEL RESULT")
print("Accuracy:", accuracy_score(y_test, y_final))
print(classification_report(y_test, y_final))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_final))

# ==============================
# 11. APPLY MODEL TO FULL DATA
# ==============================
full_probs = model.predict_proba(X)[:,1]

df['Attrition_Predicted'] = (full_probs > FINAL_THRESHOLD).astype(int)
df['Attrition_Probability'] = full_probs

# ==============================
# 12. SAVE FINAL DATASET
# ==============================
df.to_csv("/Users/piyushdata/Documents/mysql/sql project /HR Analystic/final_hr_dashboard.csv", index=False)

print("\n✅ Final CSV saved successfully!")

🔍 Threshold Tuning Results:


===== Threshold: 0.4 =====
Accuracy: 0.5136054421768708
              precision    recall  f1-score   support

           0       0.91      0.47      0.62       247
           1       0.21      0.77      0.33        47

    accuracy                           0.51       294
   macro avg       0.56      0.62      0.48       294
weighted avg       0.80      0.51      0.57       294


===== Threshold: 0.5 =====
Accuracy: 0.6496598639455783
              precision    recall  f1-score   support

           0       0.90      0.65      0.76       247
           1       0.26      0.64      0.37        47

    accuracy                           0.65       294
   macro avg       0.58      0.65      0.56       294
weighted avg       0.80      0.65      0.70       294


===== Threshold: 0.6 =====
Accuracy: 0.7687074829931972
              precision    recall  f1-score   support

           0       0.89      0.83      0.86       247
           1       0.34      0.47    

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [26]:
df.head()

,EmployeeNumber,Age,Gender,Department,JobRole,MaritalStatus,MonthlyIncome,DistanceFromHome,YearsAtCompany,WorkLifeBalance,JobSatisfaction,attrition_flag,risk_level,distance_category,Attrition_Predicted,Attrition_Probability
0,1,41,0,2,7,2,5993,1,6,1,4,1,1,2,1,0.524625
1,2,49,1,1,6,1,5130,8,10,3,2,0,1,1,0,0.264519
2,4,37,1,1,2,2,2090,2,0,3,3,1,1,2,1,0.598109
3,5,33,0,1,6,1,2909,3,8,3,3,0,1,2,0,0.301094
4,7,27,1,1,2,1,3468,2,2,3,2,0,1,2,1,0.545063


In [27]:
df.columns

Index(['EmployeeNumber', 'Age', 'Gender', 'Department', 'JobRole',
       'MaritalStatus', 'MonthlyIncome', 'DistanceFromHome', 'YearsAtCompany',
       'WorkLifeBalance', 'JobSatisfaction', 'attrition_flag', 'risk_level',
       'distance_category', 'Attrition_Predicted', 'Attrition_Probability'],
      dtype='object')